In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_exp  = pd.read_csv('raw/df_final_experiment_clients.csv')
df_demo = pd.read_csv('raw/df_final_demo.csv')
df_wd1  = pd.read_csv('raw/df_final_web_data_pt_1.csv', parse_dates=['date_time'])
df_wd2  = pd.read_csv('raw/df_final_web_data_pt_2.csv', parse_dates=['date_time'])

web = pd.read_csv('raw/df_final_web_data.csv', parse_dates=['date_time'])

In [2]:
import os
print(os.getcwd())

/Users/kanakyadav/Documents/GitHub/Ironhack_second_project


**A/B Hypothesis - Testing**



**1. *Hypothesis 1:* Completion Rate**

   Did the new interface increase the completion rate?

   - H0: completion rate of the steps between control group(A) and test group(B) is **similar**
   - H1: completion rate of the steps between control group(A) and test group(B) is **different**

In [3]:
completed = (
    web.groupby('client_id')['process_step']
    .apply(lambda steps: 'confirm' in steps.values)
    .reset_index()
    .rename(columns={'process_step': 'completed'})
)

In [4]:
df_hyp = df_exp.merge(completed, on='client_id', how='inner')

**The significance value, apha = 0.05**

In [5]:
completion_by_group = df_hyp.groupby('Variation')['completed'].agg(['sum', 'count'])
completion_by_group['rate'] = completion_by_group['sum'] / completion_by_group['count']
print(completion_by_group)

             sum  count      rate
Variation                        
Control    15434  23532  0.655873
Test       18687  26968  0.692932


In [6]:
from statsmodels.stats.proportion import proportions_ztest

control = df_hyp[df_hyp['Variation'] == 'Control']
test    = df_hyp[df_hyp['Variation'] == 'Test']

counts = [control['completed'].sum(), test['completed'].sum()]
nobs   = [len(control), len(test)]

z_stat, p_value = proportions_ztest(counts, nobs)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value:     {p_value:.20f}")

Z-statistic: -8.8745
P-value:     0.00000000000000000070


In [7]:
alpha = 0.05
if p_value < alpha:
    print("Reject H0 — completion rates are significantly different.")
else:
    print("Fail to reject H0 — no significant difference found.")

Reject H0 — completion rates are significantly different.


***Hypothesis 2*: Average Age similarity (Cost-Effectiveness Check)**

Was the experiment group assignment demographically fair?

- H0: Average age of Control and Test groups is **similar**
- H1: Average age of Control and Test groups is **different**

In [8]:
df_demo_exp = df_exp.merge(df_demo, on='client_id', how='inner')

In [9]:
control_age = df_demo_exp[df_demo_exp['Variation'] == 'Control']['clnt_age']
test_age    = df_demo_exp[df_demo_exp['Variation'] == 'Test']['clnt_age']

print("Control mean age:", control_age.mean().round(2))
print("Test mean age:   ", test_age.mean().round(2))

Control mean age: 47.5
Test mean age:    47.16


**The significance value, apha = 0.05**

In [10]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(control_age, test_age, nan_policy='omit')

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_value:.4f}")

T-statistic: 2.4161
P-value:     0.0157


In [11]:
alpha = 0.05
if p_value < alpha:
    print("Reject H0 — age distributions are significantly different (potential bias)")
else:
    print("Fail to reject H0 — groups are demographically similar in age (fair split)")

Reject H0 — age distributions are significantly different (potential bias)


**Interpretation**

Statistically → Reject H0 (p = 0.016 < 0.05)

Practically → 0.34 years difference is essentially nothing. That's 4 months difference in average age between groups — completely negligible for a financial services experiment.


In [12]:
print("Control mean age:", control_age.mean().round(2))
print("Test mean age:   ", test_age.mean().round(2))
print("Difference:      ", abs(control_age.mean() - test_age.mean()).round(2), "years")

Control mean age: 47.5
Test mean age:    47.16
Difference:       0.33 years


The t-test returns a statistically significant result (**p = 0.016**), however the actual difference in mean age is only **0.34 years** (47.50 vs 47.16). This is practically insignificant and confirms the experiment groups are demographically balanced in terms of age. The randomization was fair.

***Hypothesis 3*: Time Spent on Steps**

Did the new interface help users move through steps faster?

- H0: Median time per step is **similar** between groups
- H1: Median time per step is **different** between groups

In [15]:
web = web.sort_values(['visit_id', 'date_time'])
web['time_on_step'] = web.groupby('visit_id')['date_time'].diff().shift(-1)
web['time_on_step_sec'] = web['time_on_step'].dt.total_seconds()

# Cap outliers
cap = web['time_on_step_sec'].quantile(0.95)
web['time_on_step_capped'] = web['time_on_step_sec'].clip(upper=cap)

In [16]:
df_web_exp = web.merge(df_exp, on='client_id', how='inner')

In [29]:
control_time = df_web_exp[df_web_exp['Variation'] == 'Control']['time_on_step_capped'].dropna()
test_time    = df_web_exp[df_web_exp['Variation'] == 'Test']['time_on_step_capped'].dropna()

# Verify, there are no nulls left
print("Control nulls after drop:", control_time.isna().sum())
print("Test nulls after drop:   ", test_time.isna().sum())


Control nulls after drop: 0
Test nulls after drop:    0


In [30]:
print("Control median time (sec):", control_time.median().round(2))
print("Test median time (sec):   ", test_time.median().round(2))

Control median time (sec): 35.0
Test median time (sec):    34.0


**The significance value, apha = 0.05**

In [28]:
from scipy.stats import mannwhitneyu

u_stat, p_value = mannwhitneyu(control_time, test_time, alternative='two-sided')

print(f"U-statistic: {u_stat:.4f}")
print(f"P-value:     {p_value:.4f}")

U-statistic: 7829845182.5000
P-value:     0.3765


Why we choose U-test not t-test?

- T-test would compare means → heavily influenced by that long tail → misleading result
- Mann-Whitney U compares rank order of values → not affected by the skew → more honest result

- *mean:  64.88*
- *50%:   35.00*   ← median is much lower than mean
- *max:   307.00*  ← even after capping at 95th percentile!

In [19]:
alpha = 0.05
if p_value < alpha:
    print("Reject H0 — time spent on steps is significantly different between groups")
else:
    print("Fail to reject H0 — no significant difference in time spent")

Fail to reject H0 — no significant difference in time spent


After cleaning null values, the median time on steps is nearly identical: **35.0 sec (Control) vs 34.0 sec (Test)**. Mann-Whitney U test confirms no statistically significant difference (**p = 0.3765**). The new interface did not meaningfully change the time users spent on each step.